# 07 - Urban greening priority (Phase 7)

## What this notebook is for

Deliverable (3) of the practicum: **urban greening priority recommendations**. It ranks
Colombo's 557 GN divisions for greening investment on five observed criteria, weighted by
an Analytic Hierarchy Process, and cross-checks that ranking against an independent
TOPSIS ranking, a 3-30-300 assessment, and the Colombo Wetland Complex.

Phase 6 ranked priority zones on a deliberately **interim** Phase-5 proxy
(`prediction.interim_priority_zones`). This replaces it. Because
`prediction.apply_greening_scenario` and `canopy_shift_predictors` take a plain zone list,
`greening.priority_zone_ids` swaps straight in and no scenario code changes.

| Criterion | Direction | Weight |
|---|---|---|
| Land surface temperature, 2020s dry season | high -> high priority | 0.319 |
| Share of the zone in UTFVI Bad/Worse/Worst | high -> high priority | 0.068 |
| Vegetation deficit (inverse NDVI) | low NDVI -> high priority | 0.184 |
| Population density (WorldPop 2020) | high -> high priority | 0.319 |
| Residents beyond 300 m of green space >= 0.5 ha | high deficit -> high priority | 0.109 |

## This notebook runs in THREE PARTS

| Part | Needs | What it does |
|---|---|---|
| **1** | Earth Engine | Probes the judgements, the land-cover coverage and every wetland source, then submits the exports |
| **2** | Nothing but Python | Ranks, compares, assesses 3-30-300, crosses wetland, and draws every figure from the downloaded files |
| **3** | Python (+ Earth Engine, optionally) | Exercises the guard, then writes every product under it |

**Part 2 runs with no Earth Engine session at all.** That is deliberate: everything that
decides a ranking is pure Python, which is what makes it unit-testable outside Colab.

## Thirteen decisions taken before writing this

| # | Decision | Why |
|---|---|---|
| D1 | The MCDA runs on **observed 2020s quantities only** | Track B never beat a no-change map, so a criterion drawn from a projected surface would inherit a model with no demonstrated allocation skill. `tests/test_notebook07.py` enforces this on this notebook's AST. |
| D2 | The UTFVI criterion is the **severe-class share**, not the zone mean | `rho(LST, zone-mean UTFVI) = +1.000000` **exactly**: UTFVI is `(Ts - Tmean)/Tmean` with a scalar `Tmean`, so a zone mean is an affine transform of a zone-mean LST. Using both would give heat its weight twice and nothing would look wrong. |
| D3 | **Percentile rank** normalises; **min-max** is a mandatory reported sensitivity; TOPSIS is run on raw values *and* on ranks | `pop_density` spans three orders of magnitude, so min-max would degenerate it. The third run is what lets METHOD be told apart from NORMALISATION. |
| D4 | Direction is applied **exactly once**, in `prepare_criteria` | After it, higher always means higher priority, which is what lets the overlay be a plain weighted sum and TOPSIS take a plain column maximum. |
| D5 | CR > 0.1 **warns** here and **refuses** at the writer | The Colab-run-3 lesson: `require_validated` turned a measured negative result into a traceback and destroyed every valid product beside it. An analyst needs to *see* inconsistent weights in order to fix them. |
| D6 | The **3** of 3-30-300 is reported `not_remotely_sensable` | It needs window orientations and tree stems. Reported as unmeasured, with five compliance categories rather than a boolean so no output can imply it was checked. |
| D7 | The wetland layer is a **union of four free sources**, each keeping its own band | No official Colombo Wetland Complex boundary exists in any free dataset - the 2018 accreditation is a Wetland *City* accreditation, not a Ramsar *Site* designation. |
| D8 | The whole MCDA re-runs at **DS level** | CLAUDE.md caveat 5. A top-N has no meaning at n=13, so the DS run reports rank correlation only. |
| D9 | Agreement with the Phase-5 proxy is reported `NOT_INDEPENDENT` | Three of five criteria overlap it and `rho(interim, LST)` is already +0.9829. The two will agree; that is not validation. |
| D10 | **No feasibility criterion** | Phase 6 measured grass+shrub+bare at **0.63 %** of the district. There is no vacant-land reservoir to rank, so a plantable-area criterion would be ~0 everywhere and would rank on classifier noise. Greening here means raising canopy *within* built cells. |
| D11 | The pairwise judgements are **the analyst's**, argued not elicited | Recorded per pair in `greening.ahp.pairwise`, with `derived_weights_reference` as a regression pin so any edit shows up in a test diff. The report must say so. |
| D12 | The 300 m rule is **Euclidean + a 1.3 detour ratio**, reported side by side | A true isochrone needs an OSM graph; the detour column bounds the error instead of merely admitting it. |
| D13 | Both user-asset hooks ship **null** | So the wetland cross is a proxy union and 3-30-300 compliance is an explicit **upper bound**. |

## Caveats that travel with every output here

1. **These are LAND SURFACE TEMPERATURES, not air temperatures.** Never relabel.
2. **The weights are judgements, not measurements.** The consistency ratio tests whether
   they are self-consistent, never whether they are right.
3. **The criteria are near-collinear over Colombo** - `rho(LST, green fraction) = -0.9147`.
   Step 14's ablation, not the weights, is what says how much the method adds over ranking
   by heat alone, and it prints its verdict whether or not that flatters the method.
4. **Every ranking is a property of the GN aggregation** (`caveats.zonal_not_pixel`).
5. **Compliance is an upper bound**: Dynamic World cannot tell a public park from a
   private garden, a cantonment or a golf course.
6. **UTFVI's reference moves with the data**, so a high severe share means "hotter than
   today's district average", not "hot" (`caveats.within_epoch_only`).

---
# PART 1 - probe, then export (needs Earth Engine)

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00-06 in this runtime)
%pip install -q -r requirements.txt

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "zonal_not_pixel", "sensitivity_reporting",
             "within_epoch_only", "euclidean_not_network",
             "mcda_weights_are_judgements"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 0 - imports, staleness guard, constants

Phase 7 adds a whole module. If the notebook runs against a checkout that predates it,
every later cell fails with `AttributeError` at a different place, and the real cause -
*local changes committed but not pushed* - is three screens up. So this cell names every
function Phase 7 introduces and refuses loudly if any is absent.

It also carries **every import Parts 2 and 3 need**. Part 2 is meant to be runnable as a
separate session with no Earth Engine at all, so it must not depend on an import that
happens to live in a Part 1 cell.

In [ ]:
# COLAB: RUN THIS CELL
# Every import Parts 1, 2 AND 3 need, in one place.
import glob
import json
import math
import time
import warnings

import ee
import numpy as np
import pandas as pd
from IPython.display import Image, display

from colombo_uhi import aoi, exports, greening, landcover, prediction, spatial_stats, uhi_metrics, viz

_required = {
    "greening": [
        # resolvers
        "resolve_level", "resolve_criteria", "criterion_names", "criterion_columns",
        "resolve_normalisation", "resolve_landcover_year", "resolve_wetland_sources",
        # AHP
        "random_index", "validate_pairwise", "pairwise_matrix",
        "principal_eigenvector", "geometric_mean_weights", "consistency_index",
        "consistency_ratio", "ahp_weights", "ahp_global_weights", "build_ahp_frame",
        "require_consistent", "ConsistencyWarning", "InconsistentJudgements",
        # criterion preparation
        "apply_direction", "percentile_rank", "min_max_scale", "z_score",
        "normalise_criterion", "land_observed_fraction", "criterion_quality_flags",
        "redistribute_weights", "prepare_criteria", "require_scored_fraction",
        # overlay
        "weighted_overlay", "mcda_scores", "rank_frame",
        # TOPSIS
        "vector_normalise", "weighted_matrix", "ideal_solutions",
        "separation_measures", "closeness_coefficient", "topsis", "topsis_scores",
        # comparison and robustness
        "spearman_rho", "compare_rankings", "rank_shift_frame",
        "criterion_correlation", "effective_dimensionality", "criterion_ablation",
        "circularity_report",
        # 3-30-300
        "green_canopy_image", "population_image", "export_green_canopy_raster",
        "export_population_raster", "read_green_canopy_raster",
        "read_population_raster", "green_patches", "qualifying_green_mask",
        "service_area_mask", "detour_distance_m", "require_integer_refinement",
        "block_mean", "canopy_fraction_by_zone", "served_population_by_zone",
        "trees_in_view_proxy", "compliance_3_30_300",
        # UTFVI severe share
        "utfvi_severe_classes", "utfvi_severe_image", "utfvi_shares_by_zone",
        # wetland
        "wetland_asset_collection", "wdpa_collection", "wetland_source_image",
        "wetland_image", "export_wetland_raster", "read_wetland_raster",
        "wetland_shares_by_zone", "wetland_adjacency", "wetland_cross",
        # output and guards
        "build_priority_frame", "top_priority_zones", "priority_zone_ids",
        "require_complete_criteria", "priority_table_metadata",
        "write_priority_table", "export_priority_table", "CriteriaIncomplete",
        "PRIORITY_COLUMNS", "COMPLIANCE_CATEGORIES", "RULE_3_STATUS",
        "NOT_INDEPENDENT", "STATUS_OK", "STATUS_INSUFFICIENT", "STATUS_BELOW_FLOOR",
    ],
    "viz": [
        "greening_caption", "greening_failure_headline", "priority_palette",
        "compliance_palette", "build_greening_priority_map_figure",
        "plot_greening_priority_map", "build_ahp_weights_figure", "plot_ahp_weights",
        "build_ranking_comparison_figure", "plot_ranking_comparison",
        "build_compliance_map_figure", "plot_compliance_map",
        "build_criterion_panel_figure", "plot_criterion_panel",
        "build_priority_table_figure", "plot_priority_table",
    ],
    "spatial_stats": [
        "export_zone_covariates", "read_zone_covariates", "dynamic_world_coverage",
        "landscape_metrics_by_zone", "contiguity_neighbours", "patch_labels",
    ],
    "prediction": ["read_priority_geometry", "work_region"],
    "exports": ["describe_tasks", "export_name", "image_to_drive", "table_to_drive"],
}
_missing = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_missing = {name: funcs for name, funcs in _missing.items() if funcs}
if _missing:
    raise RuntimeError(
        "This checkout predates Phase 7. Missing: "
        + json.dumps(_missing, indent=2)
        + "\n\nRun `git push` locally, then re-run the clone cell above. The "
        "notebook pulls from GitHub, so uncommitted or unpushed work is invisible here."
    )

# Params probe: fail here rather than three cells down on a KeyError.
_probe = params.get("greening", {})
for _key in ("criteria", "ahp", "normalisation", "topsis", "comparison", "ablation",
             "rule_3_30_300", "wetland", "top_n", "outputs", "palettes"):
    if _key not in _probe:
        raise RuntimeError(f"params.greening.{_key} is absent; params.yaml predates Phase 7")

LEVEL = greening.resolve_level(None, params)
EPOCH = str(params["greening"]["epoch"])
SOURCE = str(params["greening"]["source"])
SENSITIVITY_SOURCE = str(params["greening"]["sensitivity_source"])
SCALE_M = int(params["greening"]["scale_m"])
LC_SCALE_M = int(params["greening"]["landcover_scale_m"])
POP_SCALE_M = int(params["greening"]["population_scale_m"])
TOP_N = int(params["greening"]["top_n"])
CRITERIA = greening.criterion_names(params)
OUTPUTS = params["greening"]["outputs"]

INTERIM_DIR = "data/interim"
OUTPUT_DIR = "data/outputs"
FIGURE_DIR = "figures"
for _directory in (INTERIM_DIR, OUTPUT_DIR, FIGURE_DIR):
    os.makedirs(_directory, exist_ok=True)

print(f"level={LEVEL}  epoch={EPOCH}  source={SOURCE}  scale={SCALE_M} m  top_n={TOP_N}")
print("criteria:", CRITERIA)

## Step 1 - probe the JUDGEMENTS before probing the data

The AHP costs nothing to compute and can invalidate everything downstream, so it goes
first. A set of pairwise comparisons that is not self-consistent produces weights that
look perfectly reasonable, and the only thing that reveals it is the consistency ratio.

**This cell assesses and continues.** It never calls `require_consistent`. That is the
Colab-run-3 lesson transplanted: in Phase 6, `require_validated` raised the same exception
for "this failed" as for "this was never computed", so a *measured result* read as a crash
and took every valid product beside it. An analyst whose judgements are inconsistent needs
to see the weights in order to fix them. The refusal lives at the writer in Part 3.

Two things are printed that the consistency ratio alone will not tell you:

* the **row geometric means** beside the eigenvector - the two agreeing is itself evidence
  of consistency, and a divergence is a warning the ratio can miss;
* the **heat bloc** (`lst_hot` + `utfvi_severe_share`), because the honest answer to "are
  you counting heat twice?" is a number, not an assurance.

In [ ]:
# COLAB: RUN THIS CELL
AHP_MATRIX, AHP_NAMES = greening.pairwise_matrix(params)

print("Pairwise judgements (row is how many times more important than column):")
_display = pd.DataFrame(AHP_MATRIX, index=AHP_NAMES, columns=AHP_NAMES)
print(_display.round(3).to_string())
print()

with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter("always")
    AHP_REPORT = greening.ahp_weights(AHP_MATRIX, params, AHP_NAMES)
AHP_FRAME = greening.build_ahp_frame(AHP_REPORT, params)
WEIGHTS = AHP_REPORT["weights"]

print(AHP_FRAME[["criterion", "direction", "weight", "weight_geometric"]].to_string(index=False))
print()
print(f"lambda_max           {AHP_REPORT['lambda_max']:.6f}")
print(f"consistency index    {AHP_REPORT['consistency_index']:.6f}")
print(f"random index (n={AHP_REPORT['n']})   {AHP_REPORT['random_index']:.2f}")
print(f"max eigen-vs-geomean {AHP_REPORT['max_geometric_departure']:.6f}")
print(f"weight spread        {AHP_REPORT['weight_spread']:.4f}"
      f"  (floor {AHP_REPORT['min_weight_spread']:.2f})")
print()

_bar = "=" * 72
if AHP_REPORT["consistent"] and not AHP_REPORT["degenerate"]:
    print(_bar)
    print(f"PASS: consistency ratio {AHP_REPORT['consistency_ratio']:.4f} <= "
          f"{AHP_REPORT['consistency_ratio_max']:.2f}")
    print(_bar)
else:
    print(_bar)
    print(f"*** INCONSISTENT JUDGEMENTS: CR {AHP_REPORT['consistency_ratio']:.4f} against a "
          f"{AHP_REPORT['consistency_ratio_max']:.2f} threshold ***")
    print("This is a RESULT, not a crash. The notebook continues so the weights can be")
    print("inspected and the judgements revised. Part 3's writer will REFUSE to publish")
    print("anything built on them.")
    print(_bar)
for _warning in _caught:
    print("WARNING:", _warning.message)

_heat = WEIGHTS["lst_hot"] + WEIGHTS["utfvi_severe_share"]
print()
print(f"HEAT BLOC (lst_hot + utfvi_severe_share) = {_heat:.4f} of the total weight.")
print("Stated openly: the UTFVI criterion is the SEVERE-CLASS SHARE precisely because")
print("rho(LST, zone-mean UTFVI) = +1.000000 exactly, and a mean would count heat twice.")

_hierarchy = greening.ahp_global_weights(params["greening"]["ahp"]["hierarchy"], params)
print()
print("Hierarchy sensitivity (equal groups, equal within group) - reported BESIDE the")
print("flat weights, never instead of them:")
for _name in AHP_NAMES:
    print(f"  {_name:24s} flat {WEIGHTS[_name]:.4f}   grouped {_hierarchy[_name]:.4f}")

## Step 2 - probe land-cover coverage before choosing a year

Phase 5 measured Dynamic World "green" growing **5.4x** from 2016 to 2024 over this
district. Almost all of it was Sentinel-2 coverage: 2016 classified only 10.5 % of the
export, so unclassified pixels were being counted as *not green*. Dynamic World opens
2015-06-27 and Sentinel-2B did not launch until 2017-03.

This re-probes before anything is exported, and refuses a year the classifier barely saw.

It also does something Phase 5 did not. `spatial_stats.landscape.min_observed_fraction`
compares classified area against the **polygon**, and over Colombo that measures the
polygon enclosing *water*, not classifier coverage: the identical `observed_fraction`
appears for three classifiers across three dates in **552 of 557** zones. Fort's COD-AB
polygon *is* the Colombo Port outer harbour. Excluding on that flag deletes Pettah and
Lunupokuna from the priority list - dense, hot, treeless CMC-core divisions, which is
exactly what this analysis exists to find. Step 9 recomputes the floor against **land**;
this cell prints how far apart the two are.

In [ ]:
# COLAB: RUN THIS CELL
WORK_REGION = prediction.work_region(params)
_area_km2 = WORK_REGION.area(maxError=10).divide(1e6).getInfo()
_expected = float(params["aoi"]["expected_areas_km2"]["district"])
if abs(_area_km2 - _expected) / _expected > 0.20:
    raise RuntimeError(
        f"work region is {_area_km2:.1f} km2 but Colombo District is ~{_expected:.0f} km2. "
        "Phase 6 run 1 analysed 18,090 km2 of buffered Western Province by mistake and "
        "every number it produced was void. Fix the region before exporting anything."
    )
print(f"PASS: work region {_area_km2:.1f} km2, "
      f"{100 * (_area_km2 - _expected) / _expected:+.1f}% from the expected {_expected:.0f} km2")
print()

COVERAGE = spatial_stats.dynamic_world_coverage(params, WORK_REGION)
print(COVERAGE.to_string(index=False))
print()

LC_YEAR = greening.resolve_landcover_year(params)
_row = COVERAGE.loc[COVERAGE["year"] == LC_YEAR]
if _row.empty:
    raise RuntimeError(f"the coverage probe returned no row for {LC_YEAR}")
_observed = float(_row["observed_fraction"].iloc[0])
_floor = float(params["spatial_stats"]["landscape"]["min_observed_fraction"])
if _observed < _floor:
    raise RuntimeError(
        f"Dynamic World {LC_YEAR} covers only {_observed:.1%} of the district, below the "
        f"{_floor:.0%} floor. Pick the latest year the probe clears and record it in "
        "greening.landcover_year - a greening recommendation cannot rest on a year the "
        "classifier barely saw."
    )
print(f"PASS: Dynamic World {LC_YEAR} covers {_observed:.2%} of the district "
      f"(floor {_floor:.0%}); using it for every land-cover product.")

## Step 3 - probe every wetland source before exporting one

Colombo is a Ramsar **Wetland City**, accredited 2018 - an accreditation of the *city*,
not a Ramsar *Site* designation, so there is no official polygon to download and none
exists in any free dataset. The wetland layer here is therefore a union of four free
sources, each keeping its own band so that a zone's status can say **which** evidence
fired for it.

That distinction is not decorative. WDPA is a legal designation and the other three are
remote-sensing proxies, and a policy recommendation must not present them as the same kind
of statement. This cell prints the WDPA features it finds by name, so you can see whether
Bellanwila-Attidiya is actually there before trusting the layer.

`greening.wetland.asset` is the hook for the official boundary if you can obtain it from
the Colombo Wetland Management Strategy. While it is null the source is skipped with a
printed note; requesting it explicitly raises with upload instructions.

In [ ]:
# COLAB: RUN THIS CELL
_configured = list(params["greening"]["wetland"]["sources"])
USABLE_WETLAND_SOURCES = []

if params["greening"]["wetland"].get("asset"):
    print(f"Official wetland boundary asset configured: {params['greening']['wetland']['asset']}")
else:
    print("NOTE: greening.wetland.asset is null, so no OFFICIAL Colombo Wetland Complex")
    print("      boundary is used. The cross below runs on a union of remote-sensing")
    print("      proxies plus WDPA, and every wetland result must be reported as such.")
print()

for _source in _configured:
    try:
        _image = greening.wetland_source_image(_source, params, year=LC_YEAR, region=WORK_REGION)
        _km2 = (
            _image.multiply(ee.Image.pixelArea())
            .reduceRegion(
                reducer=ee.Reducer.sum(), geometry=WORK_REGION,
                scale=LC_SCALE_M, maxPixels=int(1e10), tileScale=4,
            )
            .getInfo()
        )
        _value = float(list(_km2.values())[0] or 0.0) / 1e6
    except Exception as _error:
        print(f"  {_source:24s} FAILED: {_error}")
        continue
    print(f"  {_source:24s} {_value:9.2f} km2 inside Colombo District")
    if _value > 0:
        USABLE_WETLAND_SOURCES.append(_source)
    else:
        print(f"  {'':24s} -> returns nothing here; dropped from the union")

print()
_wdpa_all = ee.FeatureCollection(
    params["greening"]["wetland"]["source_definitions"]["wdpa"]["id"]
).filter(ee.Filter.eq("ISO3", "LKA")).filterBounds(WORK_REGION)
_name_property = params["greening"]["wetland"]["source_definitions"]["wdpa"]["name_property"]
_desig_property = params["greening"]["wetland"]["source_definitions"]["wdpa"]["designation_property"]
_keep = params["greening"]["wetland"]["source_definitions"]["wdpa"].get("designations_include")

_names = _wdpa_all.aggregate_array(_name_property).getInfo()
_designations = _wdpa_all.aggregate_array(_desig_property).getInfo()
print(f"WDPA protected areas intersecting Colombo District ({len(_names)}):")
print()
print("  INCLUDED as wetland:")
_n_included = 0
for _name, _designation in zip(_names, _designations):
    if _keep is None or _designation in _keep:
        print(f"    + {_name}  [{_designation}]")
        _n_included += 1
if _n_included == 0:
    print("    (none)")
print()
print("  EXCLUDED - protected, but not wetland:")
for _name, _designation in zip(_names, _designations):
    if _keep is not None and _designation not in _keep:
        print(f"    - {_name}  [{_designation}]")
print()
print("  *** WDPA is a PROTECTED-AREA layer, not a wetland layer. The filter is")
print("  *** greening.wetland.source_definitions.wdpa.designations_include; set it")
print("  *** to null to include every protected area.")

# *** A CONTRADICTION CHECK, NOT A FORMALITY. ***
# Run 1 listed ten protected areas by name and then reported the WDPA raster as
# 0.00 km2, so the source was dropped as "returns nothing here" while the probe
# beside it was naming every wetland site the cross exists to find. The cause was
# reduceToImage returning a fully-masked image. An empty raster that reads as an
# honest zero is the worst failure mode in this module, so it now raises.
if _n_included > 0 and "wdpa" not in USABLE_WETLAND_SOURCES:
    raise RuntimeError(
        f"{_n_included} WDPA area(s) intersect Colombo District and passed the "
        "designation filter, but the WDPA raster summed to 0 km2 and was dropped. "
        "Features exist and the raster is empty: that is a RASTERISATION fault, not "
        "an absence of wetland. Do not proceed - the wetland cross would silently "
        "run without its only legally-designated source."
    )

print()
print("USABLE_WETLAND_SOURCES:", USABLE_WETLAND_SOURCES)


## Step 4 - submit the per-zone covariate tables

Three exports, and the third is the one that matters most.

`landsat_dry` is the **pooled** L5+L7+L8+L9 series. Phase 4 forbids fitting a *trend*
across it - the inter-sensor steps over Colombo are +1.78 and -2.48 degC - but every
criterion here is a **within-epoch level**, and the ranking is over zones within one
epoch, so a spatially uniform sensor step shifts every zone equally and cancels out of the
ranking exactly as it cancels in SUHII and in Phase 5's Gi*.

That is an argument, so it is also **measured**: the third export re-runs the covariates
on the single-sensor `landsat_oli_dry` series, and Step 12 reports the rank correlation
between the two rankings. If the ranking moves, the argument is wrong for a level
criterion and the single-sensor ranking is what publishes.

Note the explicit `suffix=` on the sensitivity export. Without it the two land on the same
filename in Drive and one silently overwrites the other.

In [ ]:
# COLAB: RUN THIS CELL
TASKS = []

TASKS.append(spatial_stats.export_zone_covariates(
    params, level=LEVEL, epoch=EPOCH, region=WORK_REGION, source=SOURCE, scale_m=SCALE_M
))
TASKS.append(spatial_stats.export_zone_covariates(
    params, level="ds", epoch=EPOCH, region=WORK_REGION, source=SOURCE, scale_m=SCALE_M
))
# *** suffix= is LOAD-BEARING: without it this overwrites the primary export. ***
TASKS.append(spatial_stats.export_zone_covariates(
    params, level=LEVEL, epoch=EPOCH, region=WORK_REGION,
    source=SENSITIVITY_SOURCE, scale_m=SCALE_M, suffix=f"{LEVEL}_{EPOCH}_oli",
))

for _task in TASKS:
    print("submitted:", _task.config.get("description", _task))

## Step 5 - submit the UTFVI severe-class shares

**This is the export that exists because a zone mean would have been wrong.**

`UTFVI = (Ts - Tmean) / Tmean` with a **scalar** `Tmean` (`uhi.utfvi.reference` is
`per_year_aoi_mean`). So a zone-mean UTFVI is an affine transform of a zone-mean LST, and
measured across all 557 GN divisions the two correlate at **+1.000000 exactly**. Including
both as criteria would give heat its weight twice, and nothing on any output would look
wrong.

The severe-class **share** - the fraction of the zone's pixels in Bad, Worse or Worst - is
a within-zone distributional property that a mean cannot reproduce. The mean of a 0/1 mask
over a zone *is* that share, which is why this reuses `uhi_metrics.zonal_by_division` with
a mean reducer rather than growing a second zonal-statistics path.

In [ ]:
# COLAB: RUN THIS CELL
# This is a SYNCHRONOUS reduceRegions, not a batch export: 557 zones x one band is
# well inside the interactive limit, and it keeps the criterion table in one place
# rather than adding a raster that Part 2 would have to re-reduce.
_severe = greening.utfvi_severe_classes(params)
_labels = uhi_metrics.utfvi_class_labels(params)
print("UTFVI classes counted as severe:",
      [f"{index}={_labels[index]}" for index in _severe])
print()

UTFVI_SHARES = {}
for _level in params["greening"]["levels"]:
    _frame = greening.utfvi_shares_by_zone(
        params, level=_level, epoch=EPOCH, region=WORK_REGION, source=SOURCE, scale_m=SCALE_M
    )
    UTFVI_SHARES[_level] = _frame
    _path = os.path.join(INTERIM_DIR, f"greening_utfvi_severe_{_level}.csv")
    _frame.to_csv(_path, index=False)
    print(f"{_level}: {len(_frame)} zones, severe share "
          f"{_frame['utfvi_severe_share'].min():.3f}-{_frame['utfvi_severe_share'].max():.3f}"
          f"  -> {_path}")
print()
print("Reminder: UTFVI's reference is that epoch's own spatial mean, so a high severe")
print("share means HOTTER THAN TODAY'S DISTRICT AVERAGE, not hot in absolute terms.")

## Step 6 - submit the 10 m green / canopy raster

Three bands, and the third is not bookkeeping.

`green` is the 3-30-300 green-space class set; `canopy` is the **tree class alone**,
because the 30 % target is a *canopy* target and grass is not canopy; `observed` is 1 only
where the classifier actually produced a value inside the region.

A 0/1 raster written to GeoTIFF cannot distinguish "classified, not green" from "never
classified" - masked pixels are written as 0. That is the bug behind Phase 5's 5.4x green
growth and Phase 6's nodata-reads-as-water, and `read_green_canopy_raster` **raises** on a
band count other than three rather than trusting it.

In [ ]:
# COLAB: RUN THIS CELL
_task = greening.export_green_canopy_raster(params, region=WORK_REGION, year=LC_YEAR)
TASKS.append(_task)
print("submitted:", _task.config.get("description", _task))

print("bands:", list(greening.GREEN_CANOPY_BANDS))
print("  'water' is the JRC permanent-water mask, and it is the LAND DENOMINATOR")
print("  the coverage floor is taken against. Without it that floor measures the")
print("  polygon enclosing the harbour and excludes Pettah and Lunupokuna.")
print()

_cells = int((699 * 1e6) / (LC_SCALE_M ** 2))
print(f"~{_cells:,} cells at {LC_SCALE_M} m over Colombo District. Patch geometry and")
print("service areas are SCALE DEPENDENT, so this value is quoted with every 3-30-300 number.")

## Step 7 - submit the population raster

The 300 rule counts **residents**, not area, so the service mask has to be weighted by a
population count rather than averaged over a polygon.
`spatial_stats.population_density` returns people per km2 - the right unit for a
regression covariate and the wrong one here - so `greening.population_image` converts to
people per cell on the analysis grid.

**WorldPop ends 2020.** The year travels as a raster property and into the exported
metadata, and it must never be relabelled 2025. WorldPop is also a *modelled* surface
built partly from built-up area, which makes `pop_density` a third correlate of the same
latent factor the ablation in Step 14 measures.

In [ ]:
# COLAB: RUN THIS CELL
_population_year = int(params["spatial_stats"]["covariates"]["population"]["year"])
_task = greening.export_population_raster(params, region=WORK_REGION, year=_population_year)
TASKS.append(_task)
print("submitted:", _task.config.get("description", _task))
print(f"WorldPop year {_population_year} at {POP_SCALE_M} m. Report this year, not 2025.")

## Step 8 - submit the wetland raster

One band per source **plus** the union, a per-pixel source count, and coverage. Keeping
the sources apart is what lets Step 15 report *which* evidence fired for each zone, rather
than a single undifferentiated "wetland" flag over a legal designation and three
remote-sensing proxies.

In [ ]:
# COLAB: RUN THIS CELL
_task = greening.export_wetland_raster(
    params, region=WORK_REGION, sources=USABLE_WETLAND_SOURCES, year=LC_YEAR
)
TASKS.append(_task)
print("submitted:", _task.config.get("description", _task))
print("bands:", [*USABLE_WETLAND_SOURCES, "wetland", "n_sources", "observed"])

In [ ]:
# COLAB: RUN THIS CELL  (re-runnable - poll until every state reads COMPLETED)
_status = exports.describe_tasks(TASKS)
print(_status.to_string(index=False))
print()
print("Re-run this cell until every state reads COMPLETED, then download the products")
print("from Drive into data/interim/ before starting Part 2.")

---
# WAIT HERE

Every task above must read **COMPLETED** before Part 2 can run.

Then bring the products into `data/interim/`:

```python
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/colombo_uhi_exports/greening_* data/interim/
!cp /content/drive/MyDrive/colombo_uhi_exports/zone_covariates_* data/interim/
!ls -la data/interim/
```

**Part 2 needs no Earth Engine at all**, but it does need three cells from above. If the
runtime disconnects, or if you come back to Part 2 in a fresh session:

1. re-run the **clone** cell and the **`sys.path` / `load_params`** cell (you can let
   `init_ee()` fail - nothing in Part 2 calls Earth Engine);
2. re-run **Step 0**, for the imports and constants;
3. re-run **Step 1**, for `AHP_REPORT` / `AHP_MATRIX` / `AHP_NAMES` / `AHP_FRAME` /
   `WEIGHTS`. The AHP is pure arithmetic on `params` - it touches no data at all and no
   Earth Engine - so it costs nothing to recompute and Part 2 depends on it throughout.

Then set `LC_YEAR` and `USABLE_WETLAND_SOURCES` to the values Steps 2 and 3 printed, or
re-run those two cells with a live session.

---
# PART 2 - rank, compare and check (pure Python, no Earth Engine)

In [ ]:
# COLAB: RUN THIS CELL
# Locate every downloaded product and fail with an actionable message if absent.
def _find(pattern: str, what: str, exclude: str | None = None) -> str:
    matches = sorted(glob.glob(os.path.join(INTERIM_DIR, pattern)))
    if exclude:
        matches = [path for path in matches if exclude not in os.path.basename(path)]
    if not matches:
        raise FileNotFoundError(
            f"no {what} matching '{pattern}' in {INTERIM_DIR}/. Run Part 1, wait for "
            "every task to read COMPLETED, then copy the products out of Drive. See the "
            "WAIT HERE cell above for the copy commands."
        )
    if len(matches) > 1:
        raise FileNotFoundError(
            f"{len(matches)} files match '{pattern}' for the {what}: "
            f"{[os.path.basename(p) for p in matches]}. Picking one silently would "
            "make the ranking depend on filesystem order. Delete the stale export or "
            "tighten the pattern."
        )
    return matches[0]

# The export name template puts the LEVEL in the SUFFIX, not the product stem:
# `zone_covariates_district_2000_2025_100m_gn_2020s.csv`. Colab run 1 shipped globs
# of the form `zone_covariates_gn_*.csv`, which match none of them - so Part 2 would
# have died on a FileNotFoundError with every product sitting in data/interim/.
# The single-sensor table shares the `_gn_` marker, so it is excluded by name rather
# than by relying on sort order.
PATHS = {
    "covariates_gn": _find("zone_covariates_*_gn_*.csv", "GN covariate table", exclude="_oli"),
    "covariates_ds": _find("zone_covariates_*_ds_*.csv", "DS covariate table"),
    "covariates_oli": _find("zone_covariates_*_oli*.csv", "single-sensor covariates"),
    "utfvi_gn": _find("greening_utfvi_severe_gn.csv", "UTFVI severe shares"),
    "utfvi_ds": _find("greening_utfvi_severe_ds.csv", "UTFVI severe shares (DS)"),
    "green_canopy": _find("greening_green_canopy_*.tif", "green/canopy raster"),
    "population": _find("greening_population_*.tif", "population raster"),
    "wetland": _find("greening_wetland_*.tif", "wetland raster"),
}
for _key, _value in PATHS.items():
    print(f"{_key:16s} {os.path.basename(_value)}")


## Step 9 - assemble the criterion frame

Every join is on `zone_id` and on nothing else. **GN names are not unique within Colombo
District** - Dehiwala, Moratuwa and Kolonnawa carry divisions with the same names as CMC
ones - so joining on a name would silently merge unrelated divisions. The join losses are
printed in both directions, because a quiet inner join is how a criterion goes missing for
a fifth of the district without anyone noticing.

Then the fix that matters most in this notebook. `land_observed_fraction` recomputes the
land-cover coverage floor against **land** area rather than polygon area, and prints how
many zones change status. On the committed Phase 5 outputs the raw flag fires on 22 of
557 zones, and the identical `observed_fraction` appears for Dynamic World 2018, Dynamic
World 2024 *and* WorldCover 2021 in **552 of 557** of them. Three classifiers, three
dates, one number: it is measuring water inside the polygon. Excluding on it drops Pettah
and Lunupokuna from the priority list.

In [ ]:
# COLAB: RUN THIS CELL
COVARIATES = spatial_stats.read_zone_covariates(PATHS["covariates_gn"], params, LEVEL)
UTFVI_TABLE = pd.read_csv(PATHS["utfvi_gn"], dtype={"zone_id": str})
GEOMETRY = prediction.read_priority_geometry(
    os.path.join(OUTPUT_DIR, "gn_divisions_colombo.geojson"), params
)

COVARIATES["zone_id"] = COVARIATES["zone_id"].astype(str)
UTFVI_TABLE["zone_id"] = UTFVI_TABLE["zone_id"].astype(str)

_left = set(COVARIATES["zone_id"])
_right = set(UTFVI_TABLE["zone_id"])
print(f"covariates {len(_left)} zones, UTFVI {len(_right)} zones")
print(f"  only in covariates: {len(_left - _right)}   only in UTFVI: {len(_right - _left)}")
_geometry_ids = set(GEOMETRY["zone_id"].astype(str))
print(f"  geometry {len(_geometry_ids)} zones; not in geometry: {len(_left - _geometry_ids)}")

CRITERIA_FRAME = COVARIATES.merge(UTFVI_TABLE, on="zone_id", how="left")
print(f"joined criterion frame: {len(CRITERIA_FRAME)} zones, "
      f"{CRITERIA_FRAME.shape[1]} columns")
print()
print("Reminder [caveats.zonal_not_pixel]: every number below describes a POLYGON, not a")
print("pixel and not a person. The same analysis at DS level gives different numbers by")
print("construction, which is why Step 16 runs it.")

## Step 10 - the 3-30-300 rule

Konijnendijk's rule: **3** trees visible from every home, **30 %** canopy in every
neighbourhood, **300 m** to the nearest public green space >= 0.5 ha.

Four things about this implementation are worth stating before the numbers appear.

* **The "3" is not measurable from satellite data.** It needs window orientations and
  individual tree stems. It is reported as `not_remotely_sensable`, never enters the score
  or the verdict, and the compliance field has *five* categories rather than a boolean so
  that no output can imply it was checked.
* **"Canopy" here is the tree-class share of a 10 m modal classification**, not crown
  cover from a canopy-height model. It counts a 10 m cell as fully canopy or not at all.
* **The 300 m is Euclidean**, and the rule is about walking. Colombo puts the Kelani, Beira
  Lake, the coastal railway and walled compounds between residents and parks, so this
  *overstates* access everywhere. The 231 m detour variant is reported beside it, and the
  gap between the two columns is the size of the problem.
* **Private green counts.** Dynamic World cannot tell a public park from the Colombo Golf
  Club or an army cantonment, so every compliance number is an **upper bound**. This is
  the largest unquantified error in the phase.

The patch count is reported under 8- and 4-connectivity both, because 8-connectivity can
fuse two 0.3 ha gardens into one 0.6 ha "park" through a single diagonal pixel.

In [ ]:
# COLAB: RUN THIS CELL
BANDS, GREEN_PROFILE = greening.read_green_canopy_raster(PATHS["green_canopy"], params)
POPULATION, POP_OBSERVED, POP_PROFILE = greening.read_population_raster(
    PATHS["population"], params
)
FACTOR = int(round(POP_SCALE_M / LC_SCALE_M))

_, PATCHES_8 = greening.green_patches(
    BANDS["green"], BANDS["observed"], LC_SCALE_M, params, connectivity=8
)
_, PATCHES_4 = greening.green_patches(
    BANDS["green"], BANDS["observed"], LC_SCALE_M, params, connectivity=4
)
_min_ha = float(params["greening"]["rule_3_30_300"]["green_space"]["min_patch_ha"])
print(f"green patches >= {_min_ha} ha:  8-connectivity "
      f"{int((PATCHES_8['area_ha'] >= _min_ha).sum()):,}"
      f"   4-connectivity {int((PATCHES_4['area_ha'] >= _min_ha).sum()):,}")
print(f"total patches:            8-connectivity {PATCHES_8.attrs['n_patches']:,}"
      f"   4-connectivity {PATCHES_4.attrs['n_patches']:,}")
print()

QUALIFYING = greening.qualifying_green_mask(
    BANDS["green"], BANDS["observed"], LC_SCALE_M, params
)
SERVICE = greening.service_area_mask(QUALIFYING, LC_SCALE_M, params)
SERVICE_DETOUR = greening.service_area_mask(
    QUALIFYING, LC_SCALE_M, params, distance_m=greening.detour_distance_m(params)
)
print(f"qualifying green cells {int(QUALIFYING.sum()):,}")
print(f"served at 300 m        {int(SERVICE.sum()):,} cells")
print(f"served at {greening.detour_distance_m(params):.0f} m (detour) "
      f"{int(SERVICE_DETOUR.sum()):,} cells")
if int(SERVICE_DETOUR.sum()) > int(SERVICE.sum()):
    raise RuntimeError("the detour service area is LARGER than the plain one; check the ratio")
print()

# *** EARTH ENGINE SNAPS EACH EXPORT GRID TO ITS OWN SCALE. ***
# Run 2 stopped here: a 2957 x 4219 grid at 10 m against a 297 x 423 grid at
# 100 m, which a 10x refinement would make 2970 x 4230. The coarse grid overhangs
# by 130 m in Y and 110 m in X - one coarse cell per edge - so the two do not
# nest and block-averaging them would misregister the service mask against the
# population by up to a third of the 300 m the rule is about.
#
# The masks above were built on the FULL fine grid deliberately: the distance
# transform must see green cells beyond the trim line, or service areas would
# shrink along the new edge. Only now is everything cropped to the common extent.
ALIGNMENT = greening.align_fine_to_coarse(GREEN_PROFILE, POP_PROFILE, FACTOR, params)
print(f"grid alignment: coarse origin offset from fine by "
      f"{ALIGNMENT['origin_offset_m'][0]:+.0f} m E, "
      f"{ALIGNMENT['origin_offset_m'][1]:+.0f} m N")
print(f"  trimmed to a common extent, dropping "
      f"{ALIGNMENT['dropped_coarse_cells']:,} coarse cells "
      f"({ALIGNMENT['dropped_fraction']:.2%} = {ALIGNMENT['dropped_area_km2']:.2f} km2)")
print("  This is a BOUNDARY SNAPPING artefact, not a spatial shift: at most one")
print("  coarse cell per edge. The ceiling that keeps it a trim rather than a")
print(f"  rescue is greening.grid_alignment.max_dropped_fraction "
      f"({float(params['greening']['grid_alignment']['max_dropped_fraction']):.0%}).")

BANDS = {
    _name: greening.crop_to_window(_array, ALIGNMENT["fine_window"])
    for _name, _array in BANDS.items()
}
SERVICE = greening.crop_to_window(SERVICE, ALIGNMENT["fine_window"])
SERVICE_DETOUR = greening.crop_to_window(SERVICE_DETOUR, ALIGNMENT["fine_window"])
QUALIFYING = greening.crop_to_window(QUALIFYING, ALIGNMENT["fine_window"])
POPULATION = greening.crop_to_window(POPULATION, ALIGNMENT["coarse_window"])
POP_OBSERVED = greening.crop_to_window(POP_OBSERVED, ALIGNMENT["coarse_window"])
GREEN_PROFILE = ALIGNMENT["fine_profile"]
POP_PROFILE = ALIGNMENT["coarse_profile"]

# Zone rasters are burned onto the CROPPED grids, so every zonal statistic below
# is registered against the same ground the masks are.
ZONES_FINE, ZONE_LABELS = greening.zone_raster(GREEN_PROFILE, GEOMETRY, params)
ZONES_COARSE, _ = greening.zone_raster(POP_PROFILE, GEOMETRY, params)

# The guard that caught this in run 2 is unchanged, and now runs as a
# POST-CONDITION: alignment must satisfy it, never bypass it.
greening.require_integer_refinement(SERVICE.shape, POPULATION.shape, FACTOR)
print(f"grids register exactly: {SERVICE.shape} is {FACTOR}x {POPULATION.shape}")

CANOPY = greening.canopy_fraction_by_zone(
    BANDS["canopy"], BANDS["observed"], ZONES_FINE, ZONE_LABELS, params
)
SERVED = greening.served_population_by_zone(
    SERVICE, SERVICE_DETOUR, POPULATION, POP_OBSERVED, ZONES_COARSE,
    ZONE_LABELS, params, factor=FACTOR,
)
PROXY = greening.trees_in_view_proxy(
    BANDS["canopy"], ~BANDS["green"] & BANDS["observed"], ZONES_FINE, ZONE_LABELS, params
)
COMPLIANCE = greening.compliance_3_30_300(CANOPY, SERVED, PROXY, params)

print()
print("3-30-300 compliance, five categories (the '3' is UNMEASURED, not passed):")
for _category, _count in COMPLIANCE.attrs["counts"].items():
    print(f"  {_category:18s} {_count:4d}")
print()
print(f"median canopy share          {COMPLIANCE['canopy_pct'].median():.1f} %")
print(f"median residents within 300 m {COMPLIANCE['pop_within_300m_pct'].median():.1f} %")
print(f"median under the detour rule  {COMPLIANCE['pop_within_300m_detour_pct'].median():.1f} %")
print()
print("*** COMPLIANCE IS AN UPPER BOUND: private gardens, cantonments and golf courses")
print("*** all count as accessible green space here.")

## Step 11 - the weights, and what the criteria actually contain

The AHP was computed in Step 1; this is where the criteria themselves are examined.

**The correlation matrix is the most important output on this page.** Measured over the
committed Phase 5 outputs, `rho(LST, Dynamic World green fraction) = -0.9147` and
`rho(LST, Gi* z) = +0.9576`. WorldPop is itself modelled partly from built-up area, so
`pop_density` is a third correlate. If PC1 carries most of the variance then the MCDA is
effectively one-dimensional and the weights are largely decorative - which is a finding
about Colombo, and it belongs in the report rather than in a footnote.

In [ ]:
# COLAB: RUN THIS CELL
_landscape = pd.read_csv(
    os.path.join(OUTPUT_DIR, "landscape_metrics_green_by_gn.csv"), dtype={"zone_id": str}
)
_landscape = _landscape[
    (_landscape["scheme"] == "dynamic_world") & (_landscape["year"] == LC_YEAR)
]
# *** THE DENOMINATOR IS LAND, AND LAND IS NOT THE POLYGON. ***
# Run 3 measured what happens when this is taken against the whole rasterised
# polygon: it reproduced the Phase 5 fraction to sixteen decimal places for 555
# of 557 zones, and Pettah (0.783) and Lunupokuna (0.854) stayed excluded from
# the priority list because a fifth of each polygon is the Colombo Port outer
# harbour. `land` is the JRC permanent-water mask negated, so the harbour stops
# counting as missing data.
LAND_AREA = greening.zone_land_area(
    ZONES_FINE, ZONE_LABELS, LC_SCALE_M, observed=BANDS["land"]
)
COVERAGE_FLAGS = greening.land_observed_fraction(_landscape, LAND_AREA, params)
_water_km2 = float((~BANDS["land"]).sum()) * (LC_SCALE_M ** 2) / 1e6
print(f"permanent water inside the analysis grid: {_water_km2:.1f} km2 "
      f"(JRC occurrence >= {params['aoi']['water_mask']['jrc_occurrence_threshold_pct']}%)")

print(f"below the RAW (polygon-denominator) floor:  "
      f"{COVERAGE_FLAGS.attrs['n_below_floor_raw']:3d} zones")
print(f"below the LAND-denominator floor:           "
      f"{COVERAGE_FLAGS.attrs['n_below_floor_land']:3d} zones")
print(f"zones whose status CHANGES because of the fix: "
      f"{COVERAGE_FLAGS.attrs['n_status_changed']:3d}")
_changed = COVERAGE_FLAGS.loc[COVERAGE_FLAGS["status_changed"], "zone_id"].tolist()
if _changed:
    print("  changed:", ", ".join(_changed[:12]), "..." if len(_changed) > 12 else "")
    print("  These are divisions the polygon-denominator floor would have DELETED from the")
    print("  priority list. Fort's polygon IS the Colombo Port outer harbour.")
print()

CRITERIA_FRAME = CRITERIA_FRAME.merge(
    COVERAGE_FLAGS[["zone_id", "land_observed_fraction", "below_land_coverage_floor"]],
    on="zone_id", how="left",
)
CRITERIA_FRAME = CRITERIA_FRAME.merge(
    SERVED[["zone_id", "pop_within_300m_pct"]], on="zone_id", how="left"
)

PREPARED, PREP_REPORT = greening.prepare_criteria(CRITERIA_FRAME, params)
print(f"prepared {PREP_REPORT['n_zones']} zones: {PREP_REPORT['n_ok']} ok, "
      f"{PREP_REPORT['n_below_floor']} below floor, "
      f"{PREP_REPORT['n_insufficient']} insufficient")
print("missing per criterion:", PREP_REPORT["missing_per_criterion"])
greening.require_scored_fraction(PREPARED, params)
print()

CORRELATION = greening.criterion_correlation(PREPARED, params)
print("Criterion correlation (Spearman, on the normalised columns):")
print(CORRELATION.round(3).to_string())
print()

DIMENSIONALITY = greening.effective_dimensionality(CORRELATION)
_pc1 = DIMENSIONALITY["pc1_variance_share"]
_threshold = float(params["greening"]["ablation"]["warn_pc1_above"])
print(f"PC1 carries {_pc1:.1%} of the variance across {DIMENSIONALITY['n_criteria']} criteria")
print(f"effective dimensionality {DIMENSIONALITY['n_effective']:.2f}")
if _pc1 > _threshold:
    print()
    print("=" * 72)
    print(f"*** THE CRITERIA ARE EFFECTIVELY ONE-DIMENSIONAL (PC1 {_pc1:.1%} > "
          f"{_threshold:.0%}) ***")
    print("The AHP weights are largely decorative at this level of collinearity.")
    print("Report this alongside the ranking; Step 14 measures what it costs.")
    print("=" * 72)

## Step 12 - the weighted overlay, and two sensitivities

The ranking itself, plus the two things that would otherwise be assumed:

1. **Normalisation.** Percentile rank is primary; min-max runs beside it. If the two
   disagree, the normalisation is deciding the answer and *that* is the result.
2. **Sensor.** The whole ranking re-runs on the single-sensor `landsat_oli_dry` series. If
   the ranking moves, using the pooled series as a *level* is not safe and the
   single-sensor ranking is what publishes.

The score gap at the cut is printed because a top-60 is meaningless if ranks 60 and 61
differ in the fourth decimal.

In [ ]:
# COLAB: RUN THIS CELL
RANKED = greening.rank_frame(greening.mcda_scores(PREPARED, params, WEIGHTS), params)
print(f"ranked {len(RANKED)} zones; {RANKED.attrs['n_priority']} flagged priority "
      f"of {RANKED.attrs['n_eligible']} eligible")
print(f"score range {RANKED['score_ahp'].min():.4f} - {RANKED['score_ahp'].max():.4f}")
print(f"score gap at the cut  {RANKED.attrs['score_gap_at_cut']:.6f}")
print(f"zones tied across it  {RANKED.attrs['tied_at_cut']}")
if RANKED.attrs["score_gap_at_cut"] < 1e-3:
    print("  NOTE: the cut falls through a very flat part of the distribution. Quote the")
    print("  SCORE in the report, not only the rank.")
print()

PREPARED_MINMAX, _ = greening.prepare_criteria(CRITERIA_FRAME, params, method="min_max")
RANKED_MINMAX = greening.rank_frame(
    greening.mcda_scores(PREPARED_MINMAX, params, WEIGHTS), params
)
_normalisation = greening.compare_rankings(
    RANKED, RANKED_MINMAX, params,
    left_name="percentile rank", right_name="min-max",
    left_rank="rank_ahp", right_rank="rank_ahp",
)
print(f"NORMALISATION sensitivity: rho = {_normalisation['spearman_rho']:.4f}, "
      f"top-{TOP_N} overlap {_normalisation['top_n_overlap']}")
if _normalisation["spearman_rho"] < 0.9:
    print("  *** The normalisation is materially deciding the ranking. Report both. ***")
print()

_oli = spatial_stats.read_zone_covariates(PATHS["covariates_oli"], params, LEVEL)
_oli["zone_id"] = _oli["zone_id"].astype(str)
_oli_frame = CRITERIA_FRAME.drop(columns=["LST_C", "LST_C_pixels"], errors="ignore").merge(
    _oli[["zone_id", "LST_C", "LST_C_pixels"]], on="zone_id", how="left"
)
PREPARED_OLI, _ = greening.prepare_criteria(_oli_frame, params)
RANKED_OLI = greening.rank_frame(
    greening.mcda_scores(PREPARED_OLI, params, WEIGHTS), params
)
_sensor = greening.compare_rankings(
    RANKED, RANKED_OLI, params,
    left_name=SOURCE, right_name=SENSITIVITY_SOURCE,
    left_rank="rank_ahp", right_rank="rank_ahp",
)
print(f"SENSOR sensitivity ({SOURCE} vs {SENSITIVITY_SOURCE}): "
      f"rho = {_sensor['spearman_rho']:.4f}, top-{TOP_N} overlap {_sensor['top_n_overlap']}")
if _sensor["spearman_rho"] < 0.9:
    print("  *** The pooled series is NOT safe as a level criterion here. Publish the")
    print("  *** single-sensor ranking and say why.")
else:
    print("  The pooled-series choice is empirically justified, not merely argued.")

## Step 13 - TOPSIS, and telling the method apart from the normalisation

TOPSIS is run **twice**, and the second run is the point.

Run on raw direction-corrected values with its own vector normalisation (Hwang & Yoon
1981), TOPSIS differs from the weighted overlay in *both* method and normalisation, so a
low correlation cannot be attributed to either. Run a second time on the same percentile
ranks the overlay used, the normalisation is held constant and what remains is the method.
Three rankings, three correlations, and the decomposition is explicit.

TOPSIS is also **re-run** on the retained zone set rather than sub-selected, because its
ideal and anti-ideal come from the alternative set: removing one division can reverse the
order of two others.

In [ ]:
# COLAB: RUN THIS CELL
TOPSIS_RANKED = greening.rank_frame(
    greening.topsis_scores(PREPARED, params, WEIGHTS), params,
    score_column="score_topsis",
)
TOPSIS_ON_RANKS = greening.rank_frame(
    greening.topsis_scores(PREPARED, params, WEIGHTS, on_ranks=True), params,
    score_column="score_topsis",
)
print(f"TOPSIS drew its ideals from {TOPSIS_RANKED.attrs.get('n_alternatives', len(PREPARED))} "
      "alternatives (re-run on the retained set, never sub-selected)")
print()

COMPARISON = greening.compare_rankings(RANKED, TOPSIS_RANKED, params)
_on_ranks = greening.compare_rankings(
    RANKED, TOPSIS_ON_RANKS, params, right_name="TOPSIS on ranks"
)
_between = greening.compare_rankings(
    TOPSIS_RANKED, TOPSIS_ON_RANKS, params,
    left_name="TOPSIS (vector)", right_name="TOPSIS (ranks)",
    left_rank="rank_topsis", right_rank="rank_topsis",
)

print("THE THREE-WAY DECOMPOSITION")
print(f"  AHP overlay  vs TOPSIS (vector normalisation) : rho = "
      f"{COMPARISON['spearman_rho']:.4f}   <- method AND normalisation differ")
print(f"  AHP overlay  vs TOPSIS (on percentile ranks)  : rho = "
      f"{_on_ranks['spearman_rho']:.4f}   <- METHOD only")
print(f"  TOPSIS       vs TOPSIS (on ranks)             : rho = "
      f"{_between['spearman_rho']:.4f}   <- NORMALISATION only")
print()
print(f"  Kendall tau (AHP vs TOPSIS)  {COMPARISON['kendall_tau']:.4f}")
print(f"  top-{TOP_N} overlap            {COMPARISON['top_n_overlap']} "
      f"(Jaccard {COMPARISON['top_n_jaccard']:.3f})")
print(f"  median / max |rank shift|    {COMPARISON['median_abs_shift']:.1f} / "
      f"{COMPARISON['max_abs_shift']:.0f}")
print()

SHIFTS = greening.rank_shift_frame(RANKED, TOPSIS_RANKED, params)
print("Divisions the two methods disagree about most:")
print(SHIFTS.head(10).to_string(index=False))
print()
print("Rank agreement is a ROBUSTNESS check, not a validation: two methods run on the")
print("same five criteria agreeing says the ranking is stable under the method, not that")
print("the criteria are the right ones.")

## Step 14 - what happens when a criterion is removed

**The most valuable diagnostic in this phase**, and the one whose answer must be reported
whether or not it flatters the method.

The criteria over Colombo are near-collinear. If the full five-criterion ranking
correlates with a **land-surface-temperature-only** ranking above 0.95, then the honest
headline is *"the MCDA reproduces a ranking by LST"*, and saying so is the science.

Then the circularity check. Phase 5's interim proxy ranks on `gi_star_hot` / `lst_2020s` /
`ndvi_inverse`, and `rho(interim score, LST)` is already +0.9829. The two rankings will
agree, and that agreement is **not validation** - it is the same inputs reweighted. Phase
6's -0.84 degC counterfactual was also computed *inside* the Phase-5 zones, so quoting it
as evidence for these zones would close the loop entirely.

In [ ]:
# COLAB: RUN THIS CELL
ABLATION = greening.criterion_ablation(PREPARED, params, WEIGHTS)
print(ABLATION[["variant", "dropped", "n_criteria", "weight_dropped",
                "spearman_rho", "top_n_overlap", "max_abs_shift"]].to_string(index=False))
print()

_rho = ABLATION.attrs["single_criterion_rho"]
_threshold = ABLATION.attrs["warn_rho_above"]
_baseline = ABLATION.attrs["baseline_single_criterion"]
if ABLATION.attrs["reproduces_single_criterion"]:
    print("=" * 72)
    print(f"*** THE MCDA REPRODUCES A RANKING BY {_baseline.upper()} ALONE ***")
    print(f"rho(full five-criterion ranking, {_baseline}-only ranking) = {_rho:.4f}, "
          f"above the {_threshold:.2f} threshold.")
    print("This is a FINDING about Colombo, not a fault in the method: the criteria are")
    print("near-collinear here, so the weights move very little. It must be reported")
    print("beside the ranking, and the ranking is still the right product - it is simply")
    print("not adding what a five-criterion MCDA is normally assumed to add.")
    print("=" * 72)
else:
    print(f"The full ranking correlates {_rho:.4f} with a {_baseline}-only ranking, below")
    print(f"the {_threshold:.2f} threshold: the additional criteria do move the answer.")
print()

_interim = pd.read_csv(
    params["greening"]["ablation"]["circularity_reference"], dtype={"zone_id": str}
)
CIRCULARITY = greening.circularity_report(RANKED, _interim, params)
print(f"AGREEMENT WITH THE PHASE 5 INTERIM PROXY: rho = {CIRCULARITY['spearman_rho']:.4f}, "
      f"top-{TOP_N} overlap {CIRCULARITY['top_n_overlap']}")
print(f"independence: {CIRCULARITY['independence'].upper()}")
print()
print(" ".join(CIRCULARITY["interpretation"].split()))

## Step 15 - the wetland cross

Colombo is a Ramsar Wetland City, which makes wetland protection and expansion the
strongest local policy lever available: a high-priority division inside or beside an
existing wetland can be acted on through an instrument that already exists.

Two adjacency definitions are reported, because neither is authoritative. The **buffer**
is metric and needs a distance threshold that is frankly arbitrary, so it is computed at
250 m, 500 m and 1000 m. The **queen-neighbour** definition is topological and needs no
threshold at all. Where the two disagree is itself reported.

In [ ]:
# COLAB: RUN THIS CELL
WETLAND_BANDS, WETLAND_PROFILE = greening.read_wetland_raster(
    PATHS["wetland"], params, sources=USABLE_WETLAND_SOURCES
)
ZONES_WETLAND, _ = greening.zone_raster(WETLAND_PROFILE, GEOMETRY, params)

WETLAND_SHARES = greening.wetland_shares_by_zone(
    WETLAND_BANDS, ZONES_WETLAND, ZONE_LABELS, params, sources=USABLE_WETLAND_SOURCES
)
WETLAND_ADJ = greening.wetland_adjacency(GEOMETRY, WETLAND_SHARES, params)
WETLAND = WETLAND_ADJ.merge(
    WETLAND_SHARES.drop(columns=["wetland_within_pct"]), on="zone_id", how="left"
)

print(f"zones WITHIN wetland   {WETLAND_ADJ.attrs['n_within']}")
print(f"zones ADJACENT to it   {WETLAND_ADJ.attrs['n_adjacent']} "
      f"(buffer method at {WETLAND_ADJ.attrs['distance_m']:.0f} m)")
print(f"buffer vs queen-neighbour disagree on {WETLAND_ADJ.attrs['n_disagreement']} zones")
print()
for _distance in WETLAND_ADJ.attrs["distance_sensitivity_m"]:
    _column = f"adjacent_within_{int(_distance)}m"
    print(f"  adjacent within {int(_distance):5d} m: "
          f"{int(WETLAND_ADJ[_column].sum()):3d} zones")
print()

RANKED_WETLAND = greening.wetland_cross(RANKED, WETLAND, params)
_counts = RANKED_WETLAND.attrs.get("priority_wetland_counts", {})
print(f"Of the {RANKED_WETLAND.attrs.get('n_priority', 0)} priority divisions:")
for _status, _count in _counts.items():
    print(f"  {_status:10s} {_count:3d}")
print()
print("Which sources fired, across all zones:")
print(WETLAND_SHARES["wetland_sources"].replace("", "(none)").value_counts().to_string())
print()
if not params["greening"]["wetland"].get("asset"):
    print("*** No OFFICIAL Colombo Wetland Complex boundary was used. These are")
    print("*** remote-sensing proxies plus WDPA's legally declared areas.")

## Step 16 - the MAUP sensitivity at DS level

CLAUDE.md caveat 5. The whole ranking is a property of the 557-division aggregation: GN
divisions range from about 0.1 to 15 km2, so a top-60 is 60 **polygons**, not 60 equal
pieces of city.

At DS level there are 13 units, so a top-N has no meaning at all and none is computed.
What is reported is the correlation between a GN division's rank and its parent DS
division's rank - which is what says whether the geography survives the coarsening.

In [ ]:
# COLAB: RUN THIS CELL
_ds_covariates = spatial_stats.read_zone_covariates(PATHS["covariates_ds"], params, "ds")
_ds_utfvi = pd.read_csv(PATHS["utfvi_ds"], dtype={"zone_id": str})
_ds_covariates["zone_id"] = _ds_covariates["zone_id"].astype(str)
_ds_frame = _ds_covariates.merge(_ds_utfvi, on="zone_id", how="left")

# The access criterion has to be recomputed at DS level, not averaged up from GN:
# a population-weighted share over a DS division is not the mean of its GN shares.
DS_GEOMETRY = prediction.read_priority_geometry(
    os.path.join(OUTPUT_DIR, "ds_divisions_colombo.geojson"), params
)
DS_ZONES_FINE, DS_LABELS = greening.zone_raster(GREEN_PROFILE, DS_GEOMETRY, params)
DS_ZONES_COARSE, _ = greening.zone_raster(POP_PROFILE, DS_GEOMETRY, params)
SERVED_DS = greening.served_population_by_zone(
    SERVICE, SERVICE_DETOUR, POPULATION, POP_OBSERVED, DS_ZONES_COARSE,
    DS_LABELS, params, factor=FACTOR,
)
_ds_frame = _ds_frame.merge(
    SERVED_DS[["zone_id", "pop_within_300m_pct"]], on="zone_id", how="left"
)

DS_PREPARED, DS_REPORT = greening.prepare_criteria(_ds_frame, params)
DS_RANKED = greening.rank_frame(
    greening.mcda_scores(DS_PREPARED, params, WEIGHTS), params, top_n=len(DS_PREPARED)
)
print(f"DS level: {len(DS_RANKED)} units. A top-{TOP_N} has NO MEANING at this n and is")
print("not computed; the ranking is reported in full.")
print()
print(DS_RANKED[["rank_ahp", "zone_id", "score_ahp", "status"]].to_string(index=False))
print()

_parent = GEOMETRY[["zone_id", "adm3_pcode"]].copy() if "adm3_pcode" in GEOMETRY.columns else None
if _parent is not None:
    _parent["zone_id"] = _parent["zone_id"].astype(str)
    _linked = (
        RANKED[["zone_id", "rank_ahp"]]
        .merge(_parent, on="zone_id", how="inner")
        .merge(
            DS_RANKED[["zone_id", "rank_ahp"]].rename(
                columns={"zone_id": "adm3_pcode", "rank_ahp": "rank_ds"}
            ),
            on="adm3_pcode", how="inner",
        )
    )
    if len(_linked) > 2:
        _rho, _p = greening.spearman_rho(_linked["rank_ahp"], _linked["rank_ds"])
        print(f"rho(GN rank, parent DS rank) = {_rho:.4f} over {len(_linked)} divisions")
        print("A high value says the coarse geography preserves the fine one; a low value")
        print("says the DS aggregation has averaged the signal away. Both are results.")
else:
    print("No adm3_pcode on the geometry, so the GN-to-parent-DS link is not computed.")

## Step 17 - the figures

Five figures. Every one of them carries the consistency ratio, and every one stamps
`INCONSISTENT JUDGEMENTS` in its title if the AHP failed - because a figure showing that
judgements failed is exactly what the report needs, and refusing to draw it would throw
the evidence away.

The **criterion panel** is the one to look at hardest. Five criterion maps that all look
the same are not a redundancy in the figure; they are the finding Step 11 measured.

In [ ]:
# COLAB: RUN THIS CELL
FULL = greening.build_priority_frame(
    RANKED, params,
    prepared=PREPARED,
    topsis_ranked=TOPSIS_RANKED,
    compliance=COMPLIANCE,
    wetland=WETLAND,
    geometry=GEOMETRY.drop(columns="geometry"),
)
print(f"priority table: {len(FULL)} rows x {FULL.shape[1]} columns")
print()

FIGURES = {}
FIGURES["priority_map"] = viz.plot_greening_priority_map(
    GEOMETRY, FULL, os.path.join(FIGURE_DIR, "greening_priority_gn.png"),
    params, ahp_report=AHP_REPORT,
)
FIGURES["ahp_weights"] = viz.plot_ahp_weights(
    AHP_FRAME, AHP_REPORT, os.path.join(FIGURE_DIR, "greening_ahp_weights.png"),
    params, matrix=AHP_MATRIX, names=AHP_NAMES,
)
FIGURES["comparison"] = viz.plot_ranking_comparison(
    RANKED, TOPSIS_RANKED, COMPARISON,
    os.path.join(FIGURE_DIR, "greening_ahp_vs_topsis.png"),
    params, shifts=SHIFTS, ahp_report=AHP_REPORT,
)
FIGURES["compliance_map"] = viz.plot_compliance_map(
    GEOMETRY, COMPLIANCE, os.path.join(FIGURE_DIR, "greening_3_30_300_gn.png"),
    params, ahp_report=AHP_REPORT,
)
FIGURES["criterion_panel"] = viz.plot_criterion_panel(
    GEOMETRY, PREPARED, os.path.join(FIGURE_DIR, "greening_criteria_gn.png"),
    params, correlation=CORRELATION, ahp_report=AHP_REPORT,
)
FIGURES["priority_table"] = viz.plot_priority_table(
    FULL, os.path.join(FIGURE_DIR, "greening_priority_table.png"),
    params, ahp_report=AHP_REPORT, top_n=min(TOP_N, 30),
)

for _name, _path in FIGURES.items():
    print(f"wrote {_name:16s} {_path}")

---
# WAIT HERE (optional) - re-run the Phase 6 counterfactual on THESE zones

`greening.priority_zone_ids(FULL, params)` returns exactly the plain zone list that
`prediction.canopy_shift_predictors` already takes, so Phase 6's Step 12a can be re-run
against the Phase 7 zones without changing a line of scenario code. That is the contract
PROGRESS.md recorded when the interim proxy was introduced.

**What it buys:** deliverable 3 becomes quantitative and internally consistent - a
priority ranking *and* the cooling a 20 % canopy shift across those zones implies.

**What it is not:** validation. It is a *consequence* of the ranking, computed with the
same Track A forest on the same observed predictors. It says what would follow if these
zones were greened, not that these are the right zones.

It costs one more Earth Engine round trip plus re-running notebook 06 Step 12a with
`PRIORITY_ZONE_IDS` substituted for the interim list.

---
# PART 3 - write under the guard

## Step 18 - the guard, exercised deliberately

Before anything is written, prove the refusal works **on this runtime, with this config**.
A guard nobody has seen fire is a guard nobody knows is wired up.

Five deliberately inadequate inputs, and every one of them must be refused:

| # | Input | Why it must be refused |
|---|---|---|
| 1 | A pairwise matrix with CR far above 0.1 | The judgements contradict each other |
| 2 | A matrix of all 1s | **Perfectly consistent (CR = 0) and it has said nothing.** A passing ratio here means no judgement was made, not that a good one was |
| 3 | A non-reciprocal matrix | `a_ij * a_ji != 1` is not a comparison matrix at all |
| 4 | A table missing a criterion column | A published ranking must carry the values it was computed from |
| 5 | A table with duplicate `zone_id` | The same division would be published twice with two ranks |

In [ ]:
# COLAB: RUN THIS CELL
_checks = []

# 1 - judgements that contradict each other
_bad = np.array([[1.0, 9.0, 1 / 3], [1 / 9, 1.0, 5.0], [3.0, 1 / 5, 1.0]])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", greening.ConsistencyWarning)
    _bad_report = greening.ahp_weights(_bad, params, ["a", "b", "c"], warn=False)
_checks.append(("inconsistent judgements (CR high)",
                greening.require_consistent, (_bad_report, params),
                greening.InconsistentJudgements))

# 2 - perfectly consistent, and it decided nothing
with warnings.catch_warnings():
    warnings.simplefilter("ignore", greening.ConsistencyWarning)
    _flat_report = greening.ahp_weights(np.ones((5, 5)), params, CRITERIA, warn=False)
_checks.append((f"all-ones matrix (CR = {_flat_report['consistency_ratio']:.2f}, no judgement)",
                greening.require_consistent, (_flat_report, params),
                greening.InconsistentJudgements))

# 3 - not a comparison matrix at all
_nonrecip = np.ones((3, 3))
_nonrecip[0, 1] = 3.0
_nonrecip[1, 0] = 3.0
_checks.append(("non-reciprocal matrix",
                greening.validate_pairwise, (_nonrecip, ["a", "b", "c"]), ValueError))

# 4 - a table that cannot show its working
_no_criterion = FULL.drop(columns=["NDVI"])
_checks.append(("priority table missing a criterion column",
                greening.require_complete_criteria, (_no_criterion, params),
                greening.CriteriaIncomplete))

# 5 - the same division twice
_duplicated = pd.concat([FULL, FULL.head(1)], ignore_index=True)
_checks.append(("priority table with a duplicate zone_id",
                greening.require_complete_criteria, (_duplicated, params),
                greening.CriteriaIncomplete))

_failures = []
for _label, _function, _arguments, _expected in _checks:
    try:
        _function(*_arguments)
    except _expected as _error:
        print(f"REFUSED  {_label}")
        print(f"         -> {' '.join(str(_error).split())[:150]}")
    else:
        _failures.append(_label)
        print(f"*** NOT REFUSED *** {_label}")

print()
if _failures:
    raise RuntimeError(
        "the guard accepted inputs it must refuse: " + "; ".join(_failures)
        + ". Nothing may be written until this is fixed."
    )
print("PASS: the guard refuses every inadequate configuration.")

# And prove it lets the real product through.
greening.require_consistent(AHP_REPORT, params)
greening.require_complete_criteria(FULL, params)
print("PASS: the real AHP report and priority table clear both guards.")

## Step 19 - write every product, under the guard

`write_priority_table` calls `require_consistent` and `require_complete_criteria`
**before** it touches disk, so a refusal never leaves a half-written CSV behind for a
later cell to pick up as if it were valid. Each table travels with a `*_meta.json` sidecar
carrying the AHP report, the criterion provenance and the caveats - a ranked CSV with no
record of the judgements that produced it is not reproducible, and the weights are the
part a reader most needs and is least able to reconstruct.

In [ ]:
# COLAB: RUN THIS CELL
WRITTEN = {}

WRITTEN["ranked"] = greening.write_priority_table(
    FULL, os.path.join(OUTPUT_DIR, f"{OUTPUTS['ranked_table']}.csv"), params, AHP_REPORT,
    extra={
        "normalisation_sensitivity_rho": float(_normalisation["spearman_rho"]),
        "sensor_sensitivity_rho": float(_sensor["spearman_rho"]),
        "ahp_vs_topsis_rho": float(COMPARISON["spearman_rho"]),
        "single_criterion_rho": float(ABLATION.attrs["single_criterion_rho"]),
        "reproduces_single_criterion": bool(ABLATION.attrs["reproduces_single_criterion"]),
        "pc1_variance_share": float(DIMENSIONALITY["pc1_variance_share"]),
        "circularity_with_phase5": CIRCULARITY,
        "wetland_sources_used": USABLE_WETLAND_SOURCES,
        "official_wetland_boundary": bool(params["greening"]["wetland"].get("asset")),
        "compliance_is_upper_bound": True,
        "landcover_year": int(LC_YEAR),
    },
)

TOP = greening.top_priority_zones(FULL, params)
WRITTEN["top"] = greening.write_priority_table(
    TOP, os.path.join(OUTPUT_DIR, f"{OUTPUTS['top_table']}.csv"), params, AHP_REPORT,
    extra={"top_n": TOP_N, "n_excluded_as_flagged": int(TOP.attrs["n_excluded"])},
)

# The remaining tables are diagnostics rather than the ranked product, so they are
# written directly - but only AFTER the guard above has cleared the judgements.
for _key, _frame in (
    ("ahp_table", AHP_FRAME),
    ("compliance_table", COMPLIANCE),
    ("wetland_table", WETLAND),
    ("ablation_table", ABLATION),
    ("comparison_table", SHIFTS),
):
    _path = os.path.join(OUTPUT_DIR, f"{OUTPUTS[_key]}.csv")
    _frame.to_csv(_path, index=False)
    WRITTEN[_key] = _path

_ds_path = os.path.join(OUTPUT_DIR, f"{OUTPUTS['ranked_table'].replace('_gn', '_ds')}.csv")
DS_RANKED.to_csv(_ds_path, index=False)
WRITTEN["ranked_ds"] = _ds_path

for _name, _path in WRITTEN.items():
    print(f"wrote {_name:18s} {_path}")

PRIORITY_ZONE_IDS = greening.priority_zone_ids(FULL, params)
print()
print(f"PRIORITY_ZONE_IDS: {len(PRIORITY_ZONE_IDS)} divisions, ready for")
print("prediction.canopy_shift_predictors without any change to Phase 6's scenario code.")
print("First ten:", PRIORITY_ZONE_IDS[:10])

In [ ]:
# COLAB: RUN THIS CELL  (needs Earth Engine - skip if this session has none)
# The export name, resolved through exports.export_name so Phase 7's products are
# named the way every other phase's are - and so a name too long for an Earth Engine
# task description is REFUSED rather than truncated into a silent collision.
_name = greening.export_priority_table(FULL, params, AHP_REPORT)
print("export name:", _name)

_task = exports.table_to_drive(
    spatial_stats.division_geometry_collection(params, LEVEL),
    product=OUTPUTS["ranked_table"],
    aoi="district",
    params=params,
    file_format="GeoJSON",
)
print("submitted geometry export:", _task.config.get("description", _task))

In [ ]:
# COLAB: RUN THIS CELL
# BRING THE RESULTS HOME. Pure zipfile rather than a shell `zip`, so this runs
# anywhere and is reachable by tests/test_notebook07.py.
import zipfile

_bundle = "phase7_greening_outputs.zip"
BUNDLED = []
with zipfile.ZipFile(_bundle, "w", zipfile.ZIP_DEFLATED) as _zip:
    for _folder in (OUTPUT_DIR, FIGURE_DIR):
        for _root, _, _files in os.walk(_folder):
            for _name in _files:
                if _name == ".gitkeep" or _name.endswith(".tif"):
                    continue
                _path = os.path.join(_root, _name)
                _zip.write(_path, _path)
                BUNDLED.append(_path)

print(f"{len(BUNDLED)} file(s), {os.path.getsize(_bundle) / 1e6:.1f} MB")
for _path in sorted(BUNDLED):
    print("  ", _path)
try:
    from google.colab import files
    files.download(_bundle)
except Exception as _error:
    print(f"(auto-download unavailable: {_error} - use the Files pane)")

## What to check before signing Phase 7 off

### Must pass

| # | Check |
|---|---|
| 1 | Step 0's staleness guard passes - if it raises, `git push` and re-run the clone cell |
| 2 | Step 1 prints **CR = 0.0081** and a `PASS` box. Anything else means params and the regression pin have drifted |
| 3 | Step 2's region guard passes at ~699 km2 |
| 4 | Step 2 confirms the land-cover year clears the coverage floor |
| 5 | Step 3 finds at least one WDPA area inside Colombo District and **names** it |
| 6 | Step 10's detour service area is a strict subset of the plain one |
| 7 | Step 10's compliance summary is non-degenerate across the five categories |
| 8 | Step 11 prints how many zones change status under the land-denominator floor, and **Pettah and Lunupokuna survive** |
| 9 | Step 18 produces **five refusals**, then clears the real product |
| 10 | Step 19 writes seven CSVs plus their `_meta.json` sidecars |

### Judgement calls to REPORT, not to fix silently

| # | Question |
|---|---|
| S1 | What is `rho(full ranking, LST-only ranking)`? If it is above 0.95, **the headline is that the MCDA reproduces a ranking by LST**, and that goes in the report. |
| S2 | What share of variance does PC1 carry? Above 0.60 the weights are largely decorative. |
| S3 | Does the ranking survive the single-sensor re-run? If not, publish the `landsat_oli_dry` ranking and say why. |
| S4 | Does percentile rank agree with min-max? If not, the normalisation is deciding the answer. |
| S5 | How far apart are the 300 m and 231 m compliance columns? That gap is the size of the Euclidean-vs-network error. |
| S6 | How many priority divisions are within or beside wetland? That is the size of the policy lever. |
| S7 | Where do the buffer and queen-neighbour adjacency definitions disagree? |
| S8 | Is the score gap at the top-N cut large enough for the cut to mean anything? |

### Limits that travel into Phase 8

1. **The weights are judgements**, argued by the analyst rather than elicited from
   stakeholders. A different defensible set gives a different ranking.
2. **The criteria are near-collinear over Colombo.** The ablation is what quantifies it.
3. **Compliance is an upper bound.** Private gardens, cantonments and golf courses all
   count as accessible green space, and no free layer resolves that.
4. **The "3" of 3-30-300 was never measured**, and no output may imply otherwise.
5. **The 300 m is straight-line**, so access is overstated by an unknown amount.
6. **No official Colombo Wetland Complex boundary was used** - the layer is a union of
   three remote-sensing proxies plus WDPA's legally declared areas.
7. **Agreement with Phase 5 is not validation**, and Phase 6's counterfactual was computed
   inside the Phase-5 zones, so it cannot be cited as evidence for these.
8. **Every result is a property of the GN aggregation** (`caveats.zonal_not_pixel`).